<a href="https://colab.research.google.com/github/Ruchi0987/hospital_mapping/blob/main/hospital_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Load data
df = pd.read_csv("disease_specialty.csv")

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].astype(str).apply(clean_text)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["specialty"],
    test_size=0.2,
    random_state=42,
    stratify=df["specialty"]
)


In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC())
])

param_grid = {
    "tfidf__analyzer": ["char_wb"],
    "tfidf__ngram_range": [(3,5), (3,6)],
    "tfidf__min_df": [1, 2],
    "clf__C": [0.5, 1, 2]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                       ('clf', LinearSVC())]),
             n_jobs=-1,
             param_grid={'clf__C': [0.5, 1, 2], 'tfidf__analyzer': ['char_wb'],
                         'tfidf__min_df': [1, 2],
                         'tfidf__ngram_range': [(3, 5), (3, 6)]},
             scoring='accuracy')

In [ ]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.98
                  precision    recall  f1-score   support

      Cardiology       1.00      0.85      0.92        20
     Dermatology       1.00      1.00      1.00        20
             ENT       1.00      1.00      1.00        20
   Endocrinology       1.00      0.95      0.97        20
Gastroenterology       1.00      1.00      1.00        20
      Nephrology       1.00      1.00      1.00        20
       Neurology       0.83      1.00      0.91        20
        Oncology       1.00      1.00      1.00        20
     Orthopedics       1.00      1.00      1.00        20
     Pulmonology       1.00      1.00      1.00        20

        accuracy                           0.98       200
       macro avg       0.98      0.98      0.98       200
    weighted avg       0.98      0.98      0.98       200



In [ ]:
best_model.fit(df["text"], df["specialty"])


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5))),
                ('clf', LinearSVC(C=0.5))])

In [ ]:

def predict_specialty(user_input):
    user_input = clean_text(user_input)
    return best_model.predict([user_input])[0]

print(predict_specialty("seene me dard aur heart pain"))


Cardiology


In [ ]:
pip install sentence-transformers torch


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

class DeepDiseaseModel:
    def __init__(self):
        self.disease_df = pd.read_csv("disease_specialty.csv")
        self.hospital_df = pd.read_csv("clean_hospital_data.csv")

        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.embeddings = self.model.encode(
            self.disease_df["text"].tolist(),
            convert_to_tensor=True
        )

    def predict(self, user_text):
        query_emb = self.model.encode(user_text, convert_to_tensor=True)
        scores = util.cos_sim(query_emb, self.embeddings)
        idx = scores.argmax().item()

        specialty = self.disease_df.iloc[idx]["specialty"]
        hospitals = self.hospital_df[
            self.hospital_df["specialty"] == specialty
        ]

        return specialty, hospitals.to_dict(orient="records")
